# Composizione del dataframe (data analysis)
In questo notebook vengono modellati i dati ottenuti dal dataset [Italian Serie A (football)](https://datahub.io/football/italian-serie-a), a partire dalla stagione 2020/2021 fino alla 2025/2026. A seguito indichiamo i campi che vengono mantenuti e quelli eliminati, elencandone le motivazioni.

## Installazione delle librerie
Per l'analisi dei dati, utilizziamo le seguenti librerie:
- Pandas
- Numpy
che sono installate nell'ambiente virtuale che stiamo utilizzando, e che vengono importate nel seguente blocco

In [92]:
import numpy as np
import pandas as pd

## Importazione dataset
Nella seguente cella vengono importati i dataset che utilizziamo per l'analisi dei dati. Essendo separati li uniamo tramite la funzione di pandas "concat", e successivamente
li andiamo ad analizzare 

In [93]:
import glob
import os
import pandas as pd

# Indico la cartella nella quale sono contenuti i dataset
path_cartella = 'static/dataset'

csv_files = os.path.join(path_cartella, "*.csv")
all_files = glob.glob(csv_files)

# Creo un array di dataframe e li concateno
lista_dataframe = [pd.read_csv(file) for file in all_files]
raw_dataset = pd.concat(lista_dataframe, axis=0, ignore_index=True)

# Converto la colonna 'Date' a un formato data
raw_dataset['Date'] = pd.to_datetime(raw_dataset['Date'])

# Ordino l'intero dataset usando la colonna 'Date'
raw_dataset = raw_dataset.sort_values(by='Date', ascending=False)

# Resetto gli indici
raw_dataset = raw_dataset.reset_index(drop=True)

raw_dataset.head()


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,2026-05-24,Lecce,Genoa,1,0,H,1,0,H,NaN,...,4,1,11,6,1,6,2,0,0,0
1,2026-05-24,Cremonese,Como,1,4,A,0,1,A,NaN,...,1,6,14,12,2,4,4,2,3,0
2,2026-05-24,Parma,Sassuolo,1,0,H,0,0,D,NaN,...,3,1,10,14,1,1,2,2,0,0
3,2026-05-24,Napoli,Udinese,1,0,H,1,0,H,NaN,...,6,3,3,13,3,3,0,3,0,1
4,2026-05-24,Milan,Cagliari,1,2,A,1,1,D,NaN,...,3,10,8,9,5,5,2,2,0,0


## Rimozione dati aggiuntivi

Rimuoviamo i dati meno rilevati ai fini dell nostra analisi dal dataset. Focalizzandoci sui seguenti 
- Data partita
- Squadra locale ed ospite
- FTHG (Full Time Home Goals) FTAG (Full Time Away Goals)
- FTR (Full Time Result)
- HST (Home Shots on Target) AST (Away Shots on Target) 

In [94]:
new_index = [
    'Date',
    'HomeTeam',
    'AwayTeam',     
    'FTHG',
    'FTAG',
    'HST',
    'AST',
    'FTR'
]

clean_dataset = raw_dataset[new_index]
clean_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR
0,2026-05-24,Lecce,Genoa,1,0,4,1,H
1,2026-05-24,Cremonese,Como,1,4,1,6,A
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H
3,2026-05-24,Napoli,Udinese,1,0,6,3,H
4,2026-05-24,Milan,Cagliari,1,2,3,10,A


## Creazione di nuove colonne
Adesso tramire il dataframe rifinito e i suoi attributi, ne creiamo di nuovi, derviati, da utilizzare per l'allenamento del modello
che andremmo successivamente a generare. In particolare aggiungiamo 6 nuove categorie, ovvero:
- Stagione del campionato della riga di riferimento, in formato anno1-anno2 (es. 2023-2024);
- Media delle vittore delle ultime 5 partite;
- Differenza reti delle ultime 5 partite
- Media del rapporto $\frac{GOAL}{TIRI\_IN\_PORTA}$ delle ultime 5 partite;
- **Home advantage**, ovvero la percentuale delle vittorie in casa nelle ultime 5 partite.
- Punti cumulativi (quanti punti ha effettuato la squadra fino a quella partita esclusa)


### Stagione del campionato
Questo campo è facilmente derivabile dalla data in cui si disputa la partita: se è svolta prima di Luglio dell'anno X, allora la stagione sarà (X-1)-(X), caso contrario sarà (X)-X(X+1).

In [95]:
# Ottengo gli anni della stagione in base alla data
years = clean_dataset['Date'].dt.year
months = clean_dataset['Date'].dt.month

left_year = years.where(months > 7, years - 1)
right_year = left_year + 1

# Aggiungo la categoria "Season" al dataset
season_dataset = clean_dataset.copy()
season_dataset["Season"] = left_year.astype(str) + "-" + right_year.astype(str)

season_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR,Season
0,2026-05-24,Lecce,Genoa,1,0,4,1,H,2025-2026
1,2026-05-24,Cremonese,Como,1,4,1,6,A,2025-2026
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H,2025-2026
3,2026-05-24,Napoli,Udinese,1,0,6,3,H,2025-2026
4,2026-05-24,Milan,Cagliari,1,2,3,10,A,2025-2026


### Media vittorie e goal effettuati ultime 5 giornate

Per calcolare la media delle vittorie per squadra nelle ultime 5 partite, inizialmente creiamo colonne fittizie nel dataset per identificare le vittorie, e successivamente le raggruppiamo ad insiemi di 5 per ogni squadra.

In [21]:
import numpy as np
import pandas as pd

# Creo le colonne temporanee booleane per identificare le vittorie
season_dataset["Home_Win"] = np.where(season_dataset["FTR"] == "H", 1, 0)
season_dataset["Away_Win"] = np.where(season_dataset["FTR"] == "A", 1, 0)

# Sdoppio il dataset dal punto di vista della singola squadra
as_home = season_dataset[["Date","Season", "HomeTeam", "FTHG", "Home_Win"]].copy()
as_home.columns = ["Date","Season", "Team", "Goals", "Win"]
as_home["Home"] = 1

as_away = season_dataset[["Date","Season", "AwayTeam", "FTAG", "Away_Win"]].copy()
as_away.columns = ["Date","Season", "Team", "Goals", "Win"]
as_away["Home"] = 0

# Unisco e ordino per squadra e data
team_date = pd.concat([as_home, as_away]).sort_values(["Season","Team", "Date"])


# Calcolo le medie di interesse. La funzione rolling serve a calcolare la media mobile
# basata su una finestra temporale di massimo 5 partite precedenti.
# Lo shift invece serve a non contare la partita in esame come facente parte delle 5 precedenti.
team_date["Avg_Goals_last_5"] = (
    team_date.groupby(["Team","Season"])["Goals"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

team_date["Avg_Wins_last_5"] = (
    team_date.groupby(["Team","Season"])["Win"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# Gestisco i NaN inserendo 0 per le prime partite di campionato dove lo shift non trova dati passati
team_date["Avg_Goals_last_5"] = team_date["Avg_Goals_last_5"].fillna(0)
team_date["Avg_Wins_last_5"] = team_date["Avg_Wins_last_5"].fillna(0)

# Separo le medie calcolate per le partite in casa e per quelle in trasferta
avg_home = team_date[team_date["Home"] == 1][
    ["Date","Season", "Team", "Avg_Goals_last_5", "Avg_Wins_last_5"]
]
avg_home.columns = [
    "Date",
    "Season",
    "HomeTeam",
    "AvgH_Goals_Last5",
    "AvgH_Wins_Last5",
]

avg_away = team_date[team_date["Home"] == 0][
    ["Date","Season", "Team", "Avg_Goals_last_5", "Avg_Wins_last_5"]
]
avg_away.columns = [
    "Date",
    "Season",
    "AwayTeam",
    "AvgA_Goals_Last5",
    "AvgA_Wins_Last5",
]

avg_last5_dataset = season_dataset.copy()

# Unisco i dati appena calcolati al dataset di riferimento
avg_last5_dataset = pd.merge(avg_last5_dataset, avg_home, on=["Date","Season", "HomeTeam"], how="left")
avg_last5_dataset = pd.merge(avg_last5_dataset, avg_away, on=["Date","Season", "AwayTeam"], how="left")

# Rimuovo le colonne fittizie d'utilità create inizialmente
avg_last5_dataset = avg_last5_dataset.drop(columns=["Home_Win", "Away_Win"])

avg_last5_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR,Season,AvgH_Goals_Last5,AvgH_Wins_Last5,AvgA_Goals_Last5,AvgA_Wins_Last5
0,2026-05-24,Lecce,Genoa,1,0,4,1,H,2025-2026,1.2,0.4,0.6,0.2
1,2026-05-24,Cremonese,Como,1,4,1,6,A,2025-2026,1.0,0.4,1.0,0.6
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H,2025-2026,0.8,0.4,1.4,0.4
3,2026-05-24,Napoli,Udinese,1,0,6,3,H,2025-2026,1.8,0.4,1.4,0.4
4,2026-05-24,Milan,Cagliari,1,2,3,10,A,2025-2026,1.0,0.4,1.0,0.4


### Rapporto goal/tiri in porta
In questa cella viene calcolato il rapporto goal/tiri in porta delle (massimo) ultime 5 partite, tenendo conto che le ultime 5 partite devono essere considerate nello stesso campionato della partita indicata, non in quello precedente. Verranno aggiunti due nuovi campi, ovvero
- GoalOnShotRatioHome;
- GoalOnShotRatioAway.

Per il calcolo di questi valori dobbiamo prendere i valori delle ultime 5 partite, considerando i goal effettuati (FTAG, FTHG) e i tiri in porta (HST, AST) e calcolare i ratio delle singole partite, per poi effettuare la media tra di esse. 

In [97]:
from pandas import DataFrame
import pandas as pd

def goal_on_shot_ratio(data_frame: DataFrame) -> DataFrame:
    """
    Calcola il "goal_on_shot_ratio" medio delle ultime 5 partite per squadra e stagione.
    :params data_frame data frame dei dati
    :return data_frame contenente il GoalShotRatio
    """
    # 1. Calcolo il ratio della SINGOLA partita
    data_frame['MatchRatio'] = np.where(
        data_frame['Tiri_Porta'] > 0,
        data_frame['Gol'] / data_frame['Tiri_Porta'],
        0  
    )

    # 2. Raggruppo e faccio la MEDIA mobile dei ratio delle ultime (massimo) 5 partite
    data_frame['GoalShotRatio'] = data_frame.groupby(['Team', 'Season'])['MatchRatio'].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )
    
    # Sostituisco eventuali NaN iniziali con 0
    data_frame['GoalShotRatio'] = data_frame['GoalShotRatio'].fillna(0)

    return data_frame[['Date', 'Team', 'GoalShotRatio']]


def merge_dataframe(feature_dataframe: DataFrame, final_dataframe: DataFrame, team_name: str, ratio_name: str) -> DataFrame:
    """
    Funzione che modifica il final_dataframe, inserendo i dati del ratio indicati nel feature_dataframe
    :param feature_dataframe datagrame delle nuove caratteristiche
    :param final_dataframe dataframe in cui calcolare il risultato
    :param team_name team di riferimento (HomeTeam o AwayTeam)
    :param ratio_name indice del ratio calcolato
    :return ritorna il dataframe finale modificato
    """

    final_dataframe = pd.merge(
        final_dataframe,
        feature_dataframe,
        left_on=['Date', team_name],
        right_on=['Date', 'Team'],
        how='left'
    )
    
    # Rinominazione della colonna
    final_dataframe.rename(columns={'GoalShotRatio': ratio_name}, inplace=True)
    final_dataframe.drop('Team', axis=1, inplace=True)

    return final_dataframe


In [98]:
import numpy as np
import pandas as pd

# Colonne di utilità
utils_columns_h = ['Date', 'Season', 'HomeTeam', 'FTHG', 'HST']
utils_columns_a = ['Date', 'Season', 'AwayTeam', 'FTAG', 'AST']

# Trasformazioni colonne -> dataframe
transformation_h = {'HomeTeam': 'Team', 'FTHG': 'Gol', 'HST': 'Tiri_Porta'}
transformation_a = {'AwayTeam': 'Team', 'FTAG': 'Gol', 'AST': 'Tiri_Porta'}

# Squadra di casa
home_df = season_dataset[utils_columns_h].copy()
home_df.rename(columns=transformation_h, inplace=True)

# Squadra 
away_df = season_dataset[utils_columns_a].copy()
away_df.rename(columns=transformation_a, inplace=True)

# Unione dei dati (ordinati per data)
history = pd.concat([home_df, away_df], ignore_index=True)
history = history.sort_values(by=['Team', 'Date']).reset_index(drop=True)

utils_dataframe = goal_on_shot_ratio(history)
goal_ratio_dataframe = season_dataset.copy()

goal_ratio_dataframe = merge_dataframe(utils_dataframe, goal_ratio_dataframe, "HomeTeam", "GoalOnShotRatioHome")
goal_ratio_dataframe = merge_dataframe(utils_dataframe, goal_ratio_dataframe, "AwayTeam", "GoalOnShotRatioAway")

goal_ratio_dataframe.head()


,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR,Season,Home_Win,Away_Win,GoalOnShotRatioHome,GoalOnShotRatioAway
0,2026-05-24,Lecce,Genoa,1,0,4,1,H,2025-2026,1,0,0.250000,0.180000
1,2026-05-24,Cremonese,Como,1,4,1,6,A,2025-2026,0,1,0.190000,0.345238
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H,2025-2026,1,0,0.373333,0.295238
3,2026-05-24,Napoli,Udinese,1,0,6,3,H,2025-2026,1,0,0.288889,0.202381
4,2026-05-24,Milan,Cagliari,1,2,3,10,A,2025-2026,0,1,0.177778,0.250000


### Home advantage
Per il calcolo dell'home advantage abbiamo la necessità di trovare la percentuale delle ultime 5 partite in casa vinte, per entrambe le squadre.

In [100]:
# Riutilizzo la variabile 'team_date' già calcolata in  un blocco precedente
home_only = team_date[team_date["Home"] == 1].copy()

home_only["HomeAdvantage_Pre"] = (
    home_only.groupby("Team")["Win"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

home_only["HomeAdvantage_Post"] = (
    home_only.groupby("Team")["Win"]
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

home_advantage = home_only[["Date", "Team", "HomeAdvantage_Pre", "HomeAdvantage_Post"]].sort_values(["Team", "Date"])

home_advantage_dataset = avg_last5_dataset.copy()

home_advantage_dataset = pd.merge(
    home_advantage_dataset,
    home_advantage[["Date", "Team", "HomeAdvantage_Pre"]].rename(
        columns={"Team": "HomeTeam", "HomeAdvantage_Pre": "HomeAdvantageHome"}
    ),
    on=["Date", "HomeTeam"],
    how="left"
)

home_adv_away = pd.merge_asof(
    home_advantage_dataset[["Date", "AwayTeam"]].sort_values("Date"),
    home_advantage[["Date", "Team", "HomeAdvantage_Post"]].rename(
        columns={"Team": "AwayTeam", "HomeAdvantage_Post": "HomeAdvantageAway"}
    ).sort_values("Date"),
    on="Date",
    by="AwayTeam",
    direction="backward"
)

home_advantage_dataset = pd.merge(
    home_advantage_dataset,
    home_adv_away,
    on=["Date", "AwayTeam"],
    how="left"
)

home_advantage_dataset.head(380)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR,Season,AvgH_Goals_Last5,AvgH_Wins_Last5,AvgA_Goals_Last5,AvgA_Wins_Last5,HomeAdvantageHome,HomeAdvantageAway
0,2026-05-24,Lecce,Genoa,1,0,4,1,H,2025-2026,1.2,0.4,0.6,0.2,0.2,0.4
1,2026-05-24,Cremonese,Como,1,4,1,6,A,2025-2026,1.0,0.4,1.0,0.6,0.2,0.6
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H,2025-2026,0.8,0.4,1.4,0.4,0.2,0.6
3,2026-05-24,Napoli,Udinese,1,0,6,3,H,2025-2026,1.8,0.4,1.4,0.4,0.6,0.2
4,2026-05-24,Milan,Cagliari,1,2,3,10,A,2025-2026,1.0,0.4,1.0,0.4,0.4,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025-08-24,Cagliari,Fiorentina,1,1,6,1,D,2025-2026,1.4,0.4,1.8,0.6,0.4,0.8
376,2025-08-23,Roma,Bologna,1,0,4,2,H,2025-2026,1.6,0.8,1.0,0.0,0.8,0.4
377,2025-08-23,Milan,Cremonese,1,2,6,3,A,2025-2026,2.0,0.8,1.4,0.4,0.6,0.6
378,2025-08-23,Sassuolo,Napoli,0,2,2,4,A,2025-2026,0.8,0.2,1.4,0.6,0.2,0.8
